# Automatic Encoding for the Iceberg Code 

We have seen in the Steane code example that Guppy FT provides automatic encoding for Guppy programs. In this example, we will demonstrate automatic encoding for the $[[k+2, k, 2]]$ Iceberg code. Unlike Steane, encoding programs with the Iceberg code requires the use of Nexus cloud.

As Nexus cloud is required, we will need the `qnexus` client package.

```shell
uv add qnexus
```

In addition, we will also need a Quantinuum Nexus account. 


Once we have the `qnexus` package installed, we can login to Nexus and initialize a project to keep track of all of the jobs we run.

In [ ]:
import qnexus as qnx

qnx.login()

project = qnx.projects.get_or_create(name="iceberg_project")
qnx.context.set_active_project(project)

We can now define our computational program using Guppy. Here we define a `prepare_ghz_state` function which prepares a GHZ state over $n$ qubits. We also need a `main` function which defines the entrypoint to our quantum program.

In [7]:

from guppylang import guppy
from guppylang.std.quantum import qubit, h, cx, measure_array, collect_measurements
from guppylang.std.platform import output
from guppylang.defs import GuppyFunctionDefinition



def build_ghz_main(n_qubits: int) -> GuppyFunctionDefinition:
    @guppy
    def prepare_ghz_state() -> array[qubit, n_qubits]:
        qs = array(qubit() for _ in range(n_qubits))
        h(qs[0])
        for i in range(n_qubits - 1):
            cx(qs[i], qs[i + 1])
        return qs

    @guppy
    def main() -> None:
        ghz_qubits: array[qubit, n_qubits] = prepare_ghz_state()
        measurements = measure_array(ghz_qubits)
        output("m", collect_measurements(measurements))

    return main

We can now create a 16 qubit instance of our logical GHZ program and compile it to the HUGR IR.

In [8]:
ghz_main = build_ghz_main(16)
pkg = ghz_main.compile()

We can now use the `{py:class}`~guppyft.code.iceberg.encode.IcebergBuilder` to specify some key parameters of the Iceberg code. 

The `{py:class}`~guppyft.code.iceberg.encode.IcebergBuilder` takes an optional `{py:class}`~guppyft.code.iceberg.encode.IcebergQEDPolicy`

In [ ]:
from guppyft.code.iceberg.encode import IcebergBuilder

builder = IcebergBuilder()

In [ ]:
builder.build(block_size=2, n_blocks=8).annotate(pkg)